# Stage 1: guarded trend blend

Run `00_colab_setup.ipynb` and the full anchor baseline first. This notebook updates the repository, runs the promoted 95/4/1 fixed blend on all 773 wells, and compares it with `baseline_anchor_full_v001`. CPU is sufficient.

In [ ]:
from pathlib import Path
import json
import os
import subprocess

repo_dir = Path('/content/ROGII')
data_dir = Path('/content/rogii-data')
artifact_dir = Path('/content/drive/MyDrive/kaggle/rogii/artifacts')
assert (repo_dir / '.git').is_dir(), 'Run 00_colab_setup.ipynb first'
assert (data_dir / 'train').is_dir(), 'Competition data is not copied to /content yet'
assert (artifact_dir / 'baseline_anchor_full_v001' / 'metrics.json').is_file(), 'Run the full anchor baseline first'
os.environ['ROGII_DATA_DIR'] = str(data_dir)
os.environ['ROGII_ARTIFACT_DIR'] = str(artifact_dir)

In [ ]:
subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)

In [ ]:
RUN_ID = 'stage1_trend_blend_full_v001'
subprocess.run([
    'uv', 'run', 'rogii-train',
    '--config', 'configs/experiment/trend_blend.yaml',
    '--run-id', RUN_ID,
    '--data-dir', str(data_dir),
    '--artifact-dir', str(artifact_dir),
], cwd=repo_dir, check=True)

In [ ]:
baseline_metrics = json.loads((artifact_dir / 'baseline_anchor_full_v001' / 'metrics.json').read_text())
candidate_metrics = json.loads((artifact_dir / RUN_ID / 'metrics.json').read_text())
{
    'baseline_rmse': baseline_metrics['pooled_rmse'],
    'candidate_rmse': candidate_metrics['pooled_rmse'],
    'rmse_delta': candidate_metrics['pooled_rmse'] - baseline_metrics['pooled_rmse'],
    'candidate_metrics': candidate_metrics,
}

In [ ]:
comparison_dir = artifact_dir / 'stage1_comparison_v001'
subprocess.run([
    'uv', 'run', 'rogii-compare',
    '--baseline', str(artifact_dir / 'baseline_anchor_full_v001'),
    '--candidate', str(artifact_dir / RUN_ID),
    '--output-dir', str(comparison_dir),
], cwd=repo_dir, check=True)